# RDD 이미지 탐지 실험

원본 Kaggle 노트북을 모듈로 정리한 실행 안내입니다. 저장소를 clone하고 루트에서 `python -m pip install -r requirements.txt`를 실행한 뒤 같은 환경의 Jupyter 커널을 선택하세요.

참고 논문: [A Debiased Reconstruction-based Framework for Training-Free Detection of AI-Generated Images (CVPR 2026)](https://www.lgresearch.ai/publication/view?seq=166).

이 구현은 SD v1.4 하나를 사용하는 독립 실험이며 공식 재현이 아닙니다. 가중치를 처음 실행할 때 다운로드합니다. GPU가 권장됩니다.


In [ ]:
from pathlib import Path
import sys

# 저장소 루트 또는 notebooks 디렉터리에서 실행
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (repo_root / "rdd_detector").is_dir():
    raise RuntimeError("저장소 루트 또는 notebooks 디렉터리에서 실행하세요.")
sys.path.insert(0, str(repo_root))
from rdd_detector.detector import RDDDetector


## 모델 로딩
추가 학습은 수행하지 않습니다. noise_batch_size를 늘리면 메모리 사용량이 커집니다.

In [ ]:
detector = RDDDetector(seed=42, noise_samples=16, noise_batch_size=1)


## 로컬 이미지 점수
아래 경로를 실제 이미지로 바꾸세요. 기본은 점수만 반환합니다.

In [ ]:
image_path = repo_root / "data" / "example.jpg"
if not image_path.is_file():
    raise FileNotFoundError("image_path를 실제 이미지 경로로 변경하세요.")
result = detector.score(image_path)
result.to_dict()


## 선택적 임계값 실험
0.0001은 과거 노트북의 실험값입니다. 확률이나 보편적인 판정 기준이 아닙니다.

In [ ]:
detector.score(image_path, threshold=0.0001).to_dict()


## 결과 읽기

점수가 임계값보다 크면 `real`, 이하이면 `ai_generated`로 표시합니다. 실제 진위를 보장하는 판정이 아닙니다.

원본 출력의 20장씩 100% 결과는 `docs/experiments.md`를 참고하세요. 이번 정리 과정에서는 모델 추론을 재실행하지 않았습니다. 시드 관리와 분할 처리 등 변경으로 과거 점수와 달라질 수 있습니다.
